In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

In [ ]:
bundle = joblib.load('data/models/xgboost_production.pkl')
model = bundle['model']
threshold = bundle['threshold']

X_test = pd.read_parquet('data/processed/X_test.parquet')
Y_test = pd.read_parquet('data/processed/Y_test.parquet').iloc[:,0]

print(f"Model threshold: {threshold:.4f}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
from src.explainibility.shap_explainer import (
    build_explainer, compute_shap_values,
    get_global_importance, explain_single_prediction,
    save_explainer
)

X_train = pd.read_parquet('data/processed/X_train.parquet')
background_sample = X_train.sample(500, random_state=42)
del X_train

explainer = build_explainer(model, background_sample)
save_explainer(explainer, 'data/models/shap_explainer.pkl')

X_test_sample = X_test.sample(5000, random_state= 42)
shap_values = compute_shap_values(explainer, X_test_sample)



In [ ]:
importance = get_global_importance(
    shap_values,
    feature_names= X_test_sample.columns.tolist(),
    top_n = 20
)

print(f"\n Top 20 features by mean absolute SHAP value:")
print(importance.to_string(index=False))



In [ ]:
Y_prob = model.predict_proba(X_test)[:, 1]
high_risk_idx = Y_prob.argmax()

X_single = X_test.iloc[[high_risk_idx]]
fraud_prob = Y_prob[high_risk_idx]

print(f"Transaction fraud probability: {fraud_prob:.4f}")
print(f"Decision at threshold {threshold:.4f}: "
      f"{'FRAUD' if fraud_prob >= threshold else 'LEGITIMATE'}")

explanation = explain_single_prediction(explainer, X_single, top_n=10)

print(f"\nBase value (average prediction): {explanation['base_value']:.4f}")
print(f"This prediction:                 {explanation['prediction']:.4f}")
print(f"\nTop 10 features driving this decision:")
print(f"{'Feature':<35} {'SHAP value':>12} {'Actual value':>15}")
print("-" * 65)
for feat in explanation['top_features']:
    direction = "↑ fraud" if feat['shap_value'] > 0 else "↓ legit"
    print(f"{feat['features']:<35} "
          f"{feat['shap_value']:>+12.4f} "
          f"{str(feat['actual_value']):>15}  {direction}")